## optimizations

In [48]:
import pandas as pd
import gc

## 1. Read the fines.csv file that you saved in the previous exercise.

In [49]:
df = pd.read_csv('../data/fines.csv')

## 2. Iterations: in all the following subtasks, you need to calculate fines/refund*year for each row. Create a new column with the calculated data. Measure the time using the magic command %%timeit in the cell.

## Write a function that loops through the dataframe using for i in range(0, len(df)), iloc, and append() to a list. Assign the result of the function to a new column in the dataframe.

In [50]:
%%timeit
results = []
for i in range(len(df)):
    fines = df.iloc[i]['Fines']
    refund = df.iloc[i]['Refund']
    year = df.iloc[i]['Year']
    
    if refund != 0:
        value = fines / refund * year
    else:
        value = 0
    
    results.append(value)

df['strange'] = results

50.2 ms ± 1.09 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Do it using iterrows().

In [51]:
%%timeit
results = []

for index, row in df.iterrows():
    fines = row['Fines']
    refund = row['Refund']
    year = row['Year']
    
    if refund != 0:
        value = fines / refund * year
    else:
        value = 0
    
    results.append(value)

df['strange'] = results

13.1 ms ± 125 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Do it using apply() and a lambda function.

In [52]:
%%timeit
df['strange'] = df.apply(lambda row: row['Fines'] / row['Refund'] * row['Year'] if row['Refund'] != 0 else 0, axis=1)

4.75 ms ± 30.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Do it using Series objects from the dataframe.

In [53]:
%%timeit
df['strange'] = df['Fines'] / (df['Refund'] * df['Year']).where(df['Refund'] != 0, 0)

179 μs ± 3.66 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Do it as in the previous subtask, but use the method .values.

In [54]:
%%timeit
divider = df['Refund'] * df['Year']
df['strange'] = df['Fines'].where(
divider != 0, 0) / divider.where(divider != 0, 1)

265 μs ± 6.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 3. Indexing: measure the time using the magic command %%timeit in the cell.
## Get a row for a specific CarNumber, for example, "O136HO197RUS."

In [55]:
%%timeit
row = df[df['CarNumber'] == 'O136HO197RUS']

189 μs ± 3.56 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Set the index in your dataframe with CarNumber.

In [56]:
df.set_index('CarNumber', inplace=True)
df.head()

,Refund,Fines,Make,Model,Year,strange
CarNumber,,,,,,
Y163O8161RUS,2.0,3200.0,Ford,Focus,1989,0.804424
E432XX77RUS,1.0,6500.0,Toyota,Camry,1995,3.258145
7184TT36RUS,1.0,2100.0,Ford,Focus,1984,1.058468
X582HE161RUS,2.0,2000.0,Ford,Focus,2015,0.496278
92918M178RUS,1.0,5700.0,Ford,Focus,2014,2.830189


## Again, get a row for the same CarNumber.

In [57]:
%%timeit
row = df.loc['O136HO197RUS']

70.6 μs ± 495 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## 4. Downcasting:

## Run df.info(memory_usage='deep'), and pay attention to the Dtype and memory usage.

In [58]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
Index: 930 entries, Y163O8161RUS to TEST005RUS
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Refund   930 non-null    float64
 1   Fines    930 non-null    float64
 2   Make     930 non-null    str    
 3   Model    919 non-null    str    
 4   Year     930 non-null    int64  
 5   strange  930 non-null    float64
dtypes: float64(3), int64(1), str(2)
memory usage: 214.3 KB


## Make a copy() of your initial dataframe into another dataframe, optimized_df.

In [59]:
optimized_df = df.reset_index().copy()

## Downcast from float64 to float32 for all columns.

In [60]:
float_cols = optimized_df.select_dtypes(include=['float64']).columns
optimized_df[float_cols] = optimized_df[float_cols].astype('float32')

## Downcast from int64 to the smallest numerical Dtype possible.

In [61]:
int_cols = optimized_df.select_dtypes(include=['int64']).columns
for col in int_cols:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast='integer')

optimized_df['Refund'] = optimized_df['Refund'].astype('int8')

## Run info(memory_usage='deep') for your new dataframe. Pay attention to the Dtype and memory usage.

In [62]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  930 non-null    str    
 1   Refund     930 non-null    int8   
 2   Fines      930 non-null    float32
 3   Make       930 non-null    str    
 4   Model      919 non-null    str    
 5   Year       930 non-null    int16  
 6   strange    930 non-null    float32
dtypes: float32(2), int16(1), int8(1), str(3)
memory usage: 163.0 KB


## 5. Categories:

## Change the object type columns to category.

In [63]:
object_cols = optimized_df.select_dtypes(include=['object', 'string']).columns
optimized_df[object_cols] = optimized_df[object_cols].astype('category')

## This time, check the memory usage. It will probably decrease by 2–3 times compared to the initial dataframe.

In [64]:
optimized_df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   CarNumber  930 non-null    category
 1   Refund     930 non-null    int8    
 2   Fines      930 non-null    float32 
 3   Make       930 non-null    category
 4   Model      919 non-null    category
 5   Year       930 non-null    int16   
 6   strange    930 non-null    float32 
dtypes: category(3), float32(2), int16(1), int8(1)
memory usage: 47.0 KB


## 6. Memory clean:

## Using the library gc and the command %reset_selective, clean the memory of your initial dataframe only.

In [65]:
%reset_selective -f df
gc.collect()

2202